# In Vitro Germination and Seedling Growth Measurements for Multiple Sesame Genotypes under Controlled Temperature Regimes — Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR² sesame genotype germination dataset using `mlcroissant`.

### Dataset Source
The dataset is described using a Croissant schema accessible at the following URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
We will load the dataset metadata and contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.r5ar-n66t/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset information
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review the available record sets and their IDs, then list fields (columns) for each record set. **All Croissant entities are referenced by their `@id`.**

In [ ]:
# List all record sets
print('Available record sets:')
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# List fields for each record set
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet {rs['@id']} fields:")
    for field in rs['fields']:
        # Each field is a dict with at least '@id', maybe 'name', 'dataType', ...
        field_id = field['@id'] if isinstance(field, dict) else field
        if isinstance(field, dict):
            print(f"    @id: {field['@id']}\tname: {field.get('name', 'N/A')}\ttype: {field.get('dataType', 'N/A')}")
        else:
            print(f"    @id: {field}")

## 3. Data Extraction
Load records from each primary record set into pandas DataFrame(s) for analysis. Reference each record set by its `@id`.

In [ ]:
# Collect record set @id values (from overview above)
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Dictionary of DataFrames, one per record set @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'--- RecordSet @id: {record_set_id} ---')
    print(f'Columns: {df.columns.tolist()}')
    print(df.head(2), '\n')

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set (choose the one holding primary measurements, e.g., germination and growth data).

We'll:
- Filter for records where a numeric measurement exceeds a threshold,
- Normalize a numeric field,
- Group by genotype or experimental condition.

In [ ]:
# Select a record set to analyze (usually the largest table with numeric values)
main_record_set_id = record_set_ids[0]  # Replace [0] if preview above shows a different main table

df = dataframes[main_record_set_id]
print(f"Columns in main RecordSet: {df.columns.tolist()}")

# Select a numeric field (by @id; replace below with actual @id as needed)
# For example, '@id': 'cr:root_length', 'cr:germination_percentage', ...
# We'll choose the first numeric field for demonstration
numeric_field_id = None
# Try to select a float/int column
for c in df.columns:
    if df[c].dtype in ['float64', 'int64', 'float32', 'int32']:
        numeric_field_id = c
        break
if not numeric_field_id:
    # Try to coerce likely numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            numeric_field_id = c
            break
        except Exception:
            continue
if not numeric_field_id:
    raise Exception('No numeric field found. Please adjust the code to match your dataset field names.')

print(f"Using numeric field: {numeric_field_id}")

# Filter for values above a threshold (choose 10 as default if units fit, override as needed)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nExample normalized values:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by an identifier (e.g., genotype or temperature condition)
# Pick the first non-numeric column as group field
group_field_id = None
for c in df.columns:
    if c != numeric_field_id and df[c].dtype == object:
        group_field_id = c
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize a numeric measurement (e.g., root length or germination percentage) for each genotype or under each temperature regime.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Plot the distribution of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group if available
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded a FAIR² Croissant-packaged dataset describing in vitro germination and seedling growth in sesame genotypes,
- Explored the available record sets and field identifiers,
- Loaded the experimental data for analysis with pandas,
- Performed basic EDA: filtering, normalization, and grouping by relevant attributes,
- Visualized data distributions and attribute relationships.

This foundation enables statistical analysis, modeling, and further data-driven research on how temperature regimes affect seedling traits. For advanced analysis, apply domain-specific tests or integrate with other Croissant or FAIR datasets.